# logistic_regression_walk_forward_validation_tuning

Logistic-regression validation tuning using the full-history session-aligned dataset.

The notebook tests compact linear-model regularization settings and small feature-set variants, including price/volume baselines, alternative-data sentiment and coverage features, and attention-only GDELT/Reddit/Google feature families. Selection is based only on walk-forward validation, then selected configurations are evaluated on the untouched test split.


In [1]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings(
    "ignore",
    message=r"'penalty' was deprecated.*",
    category=FutureWarning,
    module=r"sklearn\.linear_model\._logistic",
)
warnings.filterwarnings(
    "ignore",
    message=r"Inconsistent values: penalty=.*",
    category=UserWarning,
    module=r"sklearn\.linear_model\._logistic",
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)


In [2]:
from __future__ import annotations

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebook_utils.experiment_config import build_default_config
from notebook_utils.feature_set_grid_builder import FeatureFrameBuilder, FeatureSetGridBuilder
from notebook_utils.metrics import ClassificationMetrics
from notebook_utils.model_report_builder import ModelReportBuilder
from notebook_utils.split_utils import make_split_dates, make_walk_forward_fold_specs, subset_by_dates

CONFIG = build_default_config(PROJECT_ROOT)

LOGREG_C_VALUES = [0.01, 0.1, 1.0, 10.0]


def format_grid_value(value: object) -> str:
    if value is None:
        return "none"
    if isinstance(value, float):
        return f"{value:g}".replace(".", "p")
    return str(value).replace(".", "p")


LOGREG_PARAM_GRID = []

for c_value in LOGREG_C_VALUES:
    LOGREG_PARAM_GRID.append(
        {
            "param_set": f"logreg_l2_C{format_grid_value(c_value)}_balanced_lbfgs",
            "penalty": "l2",
            "C": c_value,
            "solver": "lbfgs",
            "class_weight": "balanced",
            "l1_ratio": None,
            "max_iter": 3000,
        }
    )

for c_value in [0.1, 1.0]:
    LOGREG_PARAM_GRID.append(
        {
            "param_set": f"logreg_l2_C{format_grid_value(c_value)}_none_lbfgs",
            "penalty": "l2",
            "C": c_value,
            "solver": "lbfgs",
            "class_weight": None,
            "l1_ratio": None,
            "max_iter": 3000,
        }
    )

for c_value in [0.1, 1.0]:
    LOGREG_PARAM_GRID.append(
        {
            "param_set": f"logreg_l1_C{format_grid_value(c_value)}_balanced_saga",
            "penalty": "l1",
            "C": c_value,
            "solver": "saga",
            "class_weight": "balanced",
            "l1_ratio": None,
            "max_iter": 5000,
        }
    )

for c_value in [0.1, 1.0]:
    LOGREG_PARAM_GRID.append(
        {
            "param_set": f"logreg_l1_C{format_grid_value(c_value)}_none_saga",
            "penalty": "l1",
            "C": c_value,
            "solver": "saga",
            "class_weight": None,
            "l1_ratio": None,
            "max_iter": 5000,
        }
    )

for l1_ratio in [0.25, 0.5]:
    LOGREG_PARAM_GRID.append(
        {
            "param_set": f"logreg_elasticnet_C1_l1ratio{format_grid_value(l1_ratio)}_balanced_saga",
            "penalty": "elasticnet",
            "C": 1.0,
            "solver": "saga",
            "class_weight": "balanced",
            "l1_ratio": l1_ratio,
            "max_iter": 5000,
        }
    )

feature_grid = FeatureSetGridBuilder.build(
    max_features_per_model=CONFIG["max_features_per_model"],
)

PRICE_FEATURES = feature_grid.price_features
VOLUME_FEATURE_OPTIONS = feature_grid.volume_feature_options
GDELT_FEATURE_OPTIONS = feature_grid.gdelt_feature_options
GDELT_SENTIMENT_FEATURE_OPTIONS = feature_grid.gdelt_sentiment_feature_options
GDELT_ATTENTION_FEATURE_OPTIONS = feature_grid.gdelt_attention_feature_options
REDDIT_FEATURE_OPTIONS = feature_grid.reddit_feature_options
REDDIT_ATTENTION_FEATURE_OPTIONS = feature_grid.reddit_attention_feature_options
GOOGLE_TRENDS_FEATURE_OPTIONS = feature_grid.google_trends_feature_options
GOOGLE_SCORE_ATTENTION_FEATURE_OPTIONS = feature_grid.google_score_attention_feature_options
DERIVED_FEATURE_COLUMNS = feature_grid.derived_feature_columns
BASE_VOLUME_OPTION = feature_grid.base_volume_option
BASELINE_FEATURE_SET = feature_grid.baseline_feature_set
FEATURE_SET_SPECS = feature_grid.feature_set_specs
FEATURE_SETS = feature_grid.feature_sets
FEATURE_SET_METADATA = feature_grid.feature_set_metadata
SKIPPED_FEATURE_SETS = feature_grid.skipped_feature_sets
FEATURE_SETS_TO_TEST = feature_grid.feature_sets_to_test
pd.DataFrame(LOGREG_PARAM_GRID)


,param_set,penalty,C,solver,class_weight,l1_ratio,max_iter
0,logreg_l2_C0p01_balanced_lbfgs,l2,0.01,lbfgs,balanced,NaN,3000
1,logreg_l2_C0p1_balanced_lbfgs,l2,0.10,lbfgs,balanced,NaN,3000
2,logreg_l2_C1_balanced_lbfgs,l2,1.00,lbfgs,balanced,NaN,3000
3,logreg_l2_C10_balanced_lbfgs,l2,10.00,lbfgs,balanced,NaN,3000
4,logreg_l2_C0p1_none_lbfgs,l2,0.10,lbfgs,NaN,NaN,3000
5,logreg_l2_C1_none_lbfgs,l2,1.00,lbfgs,NaN,NaN,3000
6,logreg_l1_C0p1_balanced_saga,l1,0.10,saga,balanced,NaN,5000
7,logreg_l1_C1_balanced_saga,l1,1.00,saga,balanced,NaN,5000
8,logreg_l1_C0p1_none_saga,l1,0.10,saga,NaN,NaN,5000
9,logreg_l1_C1_none_saga,l1,1.00,saga,NaN,NaN,5000


In [3]:
from __future__ import annotations



def build_logreg_pipeline_from_params(params: dict) -> Pipeline:
    logreg_params = dict(params)
    logreg_params.pop("param_set", None)
    if logreg_params.get("penalty") != "elasticnet":
        logreg_params.pop("l1_ratio", None)
    elif pd.isna(logreg_params.get("l1_ratio")):
        logreg_params["l1_ratio"] = None
    logreg_params.setdefault("random_state", CONFIG["random_state"])
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(**logreg_params)),
        ]
    )


In [4]:
raw_df = pd.read_csv(CONFIG["dataset_path"], parse_dates=["date"])
raw_df = raw_df[~raw_df["ticker"].isin(CONFIG["excluded_tickers"])].copy()
raw_df = raw_df.sort_values(["ticker", "date"]).reset_index(drop=True)

feature_df = FeatureFrameBuilder.build_feature_frame(raw_df, neutral_band=CONFIG["neutral_band"])
train_dates, validation_dates, test_dates = make_split_dates(
    feature_df,
    test_size=CONFIG["test_size"],
    validation_fraction_within_pretest=CONFIG["validation_fraction_within_pretest"],
    min_validation_dates=CONFIG["min_validation_dates"],
    gap_days=CONFIG["gap_days"],
)
walk_forward_fold_specs = make_walk_forward_fold_specs(
    train_dates,
    n_folds=CONFIG["walk_forward_folds"],
    validation_size=CONFIG["walk_forward_validation_dates"],
    min_train_dates=CONFIG["walk_forward_min_train_dates"],
    gap_days=CONFIG["gap_days"],
)

split_date_map = {
    "train": train_dates,
    "validation": validation_dates,
    "test": test_dates,
}
feature_df["split"] = "gap"
for split_name, split_dates in split_date_map.items():
    feature_df.loc[feature_df["date"].isin(split_dates), "split"] = split_name

modeled_df = feature_df[feature_df["target"].isin(ClassificationMetrics.CLASS_VALUES)].copy()
modeled_df["target"] = modeled_df["target"].astype(int)

train_df = subset_by_dates(modeled_df, train_dates)
validation_df = subset_by_dates(modeled_df, validation_dates)
test_df = subset_by_dates(modeled_df, test_dates)
train_validation_df = subset_by_dates(modeled_df, list(train_dates) + list(validation_dates))

split_summary_rows = []
for split_name in ["train", "validation", "test"]:
    all_split_df = feature_df[feature_df["split"].eq(split_name)]
    modeled_split_df = modeled_df[modeled_df["split"].eq(split_name)]
    class_rates = modeled_split_df["target"].value_counts(normalize=True)
    split_summary_rows.append(
        {
            "split": split_name,
            "session_rows": len(all_split_df),
            "modeled_rows": len(modeled_split_df),
            "session_dates": all_split_df["date"].nunique(),
            "modeled_dates": modeled_split_df["date"].nunique(),
            "date_min": all_split_df["date"].min(),
            "date_max": all_split_df["date"].max(),
            "target_down_rate": float(class_rates.get(0, 0.0)),
            "target_neutral_rate": float(class_rates.get(1, 0.0)),
            "target_up_rate": float(class_rates.get(2, 0.0)),
        }
    )

split_summary_df = pd.DataFrame(split_summary_rows)
walk_forward_fold_summary_df = pd.DataFrame(
    [
        {
            "fold": spec["fold"],
            "train_n_dates": spec["train_n_dates"],
            "train_date_min": spec["train_date_min"],
            "train_date_max": spec["train_date_max"],
            "validation_n_dates": spec["validation_n_dates"],
            "validation_date_min": spec["validation_date_min"],
            "validation_date_max": spec["validation_date_max"],
        }
        for spec in walk_forward_fold_specs
    ]
)

split_summary_df

,split,session_rows,modeled_rows,session_dates,modeled_dates,date_min,date_max,target_down_rate,target_neutral_rate,target_up_rate
0,train,5632,5632,704,704,2021-01-04,2023-10-19,0.386541,0.208629,0.404830
1,validation,1880,1880,235,235,2023-10-23,2024-09-27,0.327128,0.242553,0.430319
2,test,2512,2504,314,313,2024-10-01,2025-12-31,0.356629,0.246805,0.396565


In [5]:
walk_forward_fold_summary_df

,fold,train_n_dates,train_date_min,train_date_max,validation_n_dates,validation_date_min,validation_date_max
0,1,383,2021-01-04,2022-07-12,80,2022-07-14,2022-11-03
1,2,463,2021-01-04,2022-11-02,80,2022-11-04,2023-03-02
2,3,543,2021-01-04,2023-03-01,80,2023-03-03,2023-06-27
3,4,623,2021-01-04,2023-06-26,80,2023-06-28,2023-10-19


In [6]:
neutral_summary_by_ticker_df = (
    feature_df.groupby("ticker")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reset_index()
)
neutral_summary_by_ticker_df["neutral_rate_among_available"] = (
    neutral_summary_by_ticker_df["neutral"] / neutral_summary_by_ticker_df["target_available"]
)
neutral_summary_by_ticker_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_ticker_df["neutral_rate_among_available"]

neutral_summary_by_split_df = (
    feature_df[feature_df["split"].isin(["train", "validation", "test"])]
    .groupby("split")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
neutral_summary_by_split_df["neutral_rate_among_available"] = (
    neutral_summary_by_split_df["neutral"] / neutral_summary_by_split_df["target_available"]
)
neutral_summary_by_split_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_split_df["neutral_rate_among_available"]

print("Neutral coverage by split")
print(neutral_summary_by_split_df.to_string(index=False))
print("\nNeutral coverage by ticker")
neutral_summary_by_ticker_df

Neutral coverage by split
     split  rows  target_available  neutral  neutral_rate_among_available  modeled_rate_among_available
     train  5632              5632     1175                      0.208629                      0.791371
validation  1880              1880      456                      0.242553                      0.757447
      test  2512              2504      618                      0.246805                      0.753195

Neutral coverage by ticker


,ticker,rows,target_available,neutral,neutral_rate_among_available,modeled_rate_among_available
0,AAPL,1255,1254,378,0.301435,0.698565
1,AMD,1255,1254,220,0.175439,0.824561
2,AMZN,1255,1254,300,0.239234,0.760766
3,GOOGL,1255,1254,319,0.254386,0.745614
4,META,1255,1254,277,0.220893,0.779107
5,MSFT,1255,1254,400,0.318979,0.681021
6,NVDA,1255,1254,173,0.137959,0.862041
7,TSLA,1255,1254,184,0.146730,0.853270


In [7]:
missing_requested_feature_sets = [name for name in FEATURE_SETS_TO_TEST if name not in FEATURE_SETS]
if missing_requested_feature_sets:
    raise KeyError(f"Unknown feature sets: {missing_requested_feature_sets}")

missing_feature_columns = sorted(
    {
        feature
        for name in FEATURE_SETS_TO_TEST
        for feature in FEATURE_SETS[name]
        if feature not in feature_df.columns and feature not in DERIVED_FEATURE_COLUMNS
    }
)
if missing_feature_columns:
    raise KeyError(f"Missing feature columns: {missing_feature_columns}")

candidate_feature_sets_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "feature_family": FEATURE_SET_METADATA[feature_set_name]["feature_family"],
            "n_features": len(features),
            "features": features,
        }
        for feature_set_name, features in FEATURE_SETS.items()
    ]
).sort_values(["feature_family", "n_features", "feature_set"]).reset_index(drop=True)

skipped_feature_sets_df = pd.DataFrame(SKIPPED_FEATURE_SETS)

print(f"Selection metric: {CONFIG['selection_metric']}")
print(f"Primary validation metric: {CONFIG['primary_validation_metric']}")
print(f"Walk-forward folds: {len(walk_forward_fold_specs)}")
print(f"Target classes: {ClassificationMetrics.CLASS_LABELS}")
print("Prediction rule: multiclass argmax")
print(f"Max features per model: {CONFIG['max_features_per_model']}")
print(f"Feature sets to test: {len(FEATURE_SETS_TO_TEST)}")
print(f"Attention feature sets: {sum('attention' in FEATURE_SET_METADATA[name]['feature_family'] for name in FEATURE_SETS_TO_TEST)}")
print(f"Skipped feature sets above max feature limit: {len(SKIPPED_FEATURE_SETS)}")
print(f"Logistic-regression parameter sets: {len(LOGREG_PARAM_GRID)}")
print(f"Walk-forward validation fits: {len(FEATURE_SETS_TO_TEST) * len(LOGREG_PARAM_GRID) * len(walk_forward_fold_specs)}")

candidate_feature_sets_df


Selection metric: balanced_accuracy
Primary validation metric: balanced_accuracy
Walk-forward folds: 4
Target classes: {0: 'down', 1: 'neutral', 2: 'up'}
Prediction rule: multiclass argmax
Max features per model: 14
Feature sets to test: 80
Attention feature sets: 38
Skipped feature sets above max feature limit: 0
Logistic-regression parameter sets: 12
Walk-forward validation fits: 3840


,feature_set,feature_family,n_features,features
0,Model B - price + volume | volume log1p zscore...,price + volume,9,"[return_1d, return_5d, return_20d, rolling_vol..."
1,Model B - price + volume | volume percentile r...,price + volume,9,"[return_1d, return_5d, return_20d, rolling_vol..."
2,Model B - price + volume | volume zscore 10d,price + volume,9,"[return_1d, return_5d, return_20d, rolling_vol..."
3,Model B - price + volume | volume zscore 20d,price + volume,9,"[return_1d, return_5d, return_20d, rolling_vol..."
4,Model B - price + volume | volume zscore 20d c...,price + volume,9,"[return_1d, return_5d, return_20d, rolling_vol..."
...,...,...,...,...
75,Model F - price + volume + all alternative dat...,price + volume + all alternative data,14,"[return_1d, return_5d, return_20d, rolling_vol..."
76,Model N - price + volume + all attention | per...,price + volume + all attention,12,"[return_1d, return_5d, return_20d, rolling_vol..."
77,Model N - price + volume + all attention | zsc...,price + volume + all attention,12,"[return_1d, return_5d, return_20d, rolling_vol..."
78,Model N - price + volume + all attention | zsc...,price + volume + all attention,12,"[return_1d, return_5d, return_20d, rolling_vol..."


In [8]:
from __future__ import annotations


LOGREG_PARAM_COLUMNS = [
    "penalty",
    "C",
    "solver",
    "class_weight",
    "l1_ratio",
    "max_iter",
]
LOGREG_PARAM_COLUMNS_FOR_DISPLAY = LOGREG_PARAM_COLUMNS


def prepare_train_eval_feature_frames(
    features: list[str],
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if "google_trends_above_ticker_train_median" in features:
        return FeatureFrameBuilder.add_google_trends_train_median_feature(train_input_df, eval_input_df)
    return train_input_df, eval_input_df


def params_from_result_row(row: dict | pd.Series) -> dict:
    l1_ratio = row.get("l1_ratio", None)
    if pd.isna(l1_ratio):
        l1_ratio = None
    class_weight = row.get("class_weight", None)
    if pd.isna(class_weight):
        class_weight = None
    return {
        "param_set": row["param_set"],
        "penalty": row["penalty"],
        "C": float(row["C"]),
        "solver": row["solver"],
        "class_weight": class_weight,
        "l1_ratio": None if l1_ratio is None else float(l1_ratio),
        "max_iter": int(row["max_iter"]),
    }

def add_param_columns(row: dict, params: dict, param_columns: list[str]) -> None:
    for column in param_columns:
        row[column] = params.get(column)


def evaluate_logreg_params(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
    split_name: str,
    return_predictions: bool = False,
) -> dict | tuple[dict, pd.DataFrame]:
    train_features_df, eval_features_df = prepare_train_eval_feature_frames(
        features,
        train_input_df,
        eval_input_df,
    )

    pipeline = build_logreg_pipeline_from_params(params)
    pipeline.fit(train_features_df[features], train_features_df["target"])
    probabilities = pipeline.predict_proba(eval_features_df[features])
    classes = pipeline.named_steps["model"].classes_
    preds = ClassificationMetrics.predictions_from_probabilities(probabilities, classes)

    metric_result = ClassificationMetrics.metrics_from_predictions(eval_features_df["target"], preds)
    preds = metric_result.pop("preds")
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    row = {
        "split": split_name,
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "param_set": params["param_set"],
        "n_features": len(features),
        **metric_result,
    }
    add_param_columns(row, params, LOGREG_PARAM_COLUMNS_FOR_DISPLAY)

    if not return_predictions:
        return row

    predictions_df = eval_features_df[["date", "ticker", "target"]].copy()
    for column, values in ClassificationMetrics.probability_column_dict(probabilities, classes).items():
        predictions_df[column] = values
    predictions_df["prediction"] = preds
    return row, predictions_df


def evaluate_logreg_config_walk_forward(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    modeled_input_df: pd.DataFrame,
    fold_specs: list[dict],
) -> tuple[dict, list[dict]]:
    fold_rows = []
    for spec in fold_specs:
        fold_row = evaluate_logreg_params(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            train_input_df=subset_by_dates(modeled_input_df, spec["train_dates"]),
            eval_input_df=subset_by_dates(modeled_input_df, spec["validation_dates"]),
            split_name="walk_forward_validation",
        )
        fold_rows.append(
            {
                **fold_row,
                "fold": spec["fold"],
                "fold_train_n_dates": spec["train_n_dates"],
                "fold_validation_n_dates": spec["validation_n_dates"],
                "fold_train_date_min": spec["train_date_min"],
                "fold_train_date_max": spec["train_date_max"],
                "fold_validation_date_min": spec["validation_date_min"],
                "fold_validation_date_max": spec["validation_date_max"],
            }
        )

    fold_results_df = pd.DataFrame(fold_rows)
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    metric_means = {
        metric: float(fold_results_df[metric].mean())
        for metric in ['accuracy', 'balanced_accuracy', 'f1_score', 'f1_weighted']
    }
    summary_row = {
        "split": "walk_forward_validation",
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "param_set": params["param_set"],
        "n_features": len(features),
        **metric_means,
    }
    add_param_columns(summary_row, params, LOGREG_PARAM_COLUMNS_FOR_DISPLAY)
    return summary_row, fold_rows

In [9]:
selection_metric = CONFIG["selection_metric"]
walk_forward_grid_rows = []
walk_forward_fold_rows = []

for feature_set_name in FEATURE_SETS_TO_TEST:
    features = FEATURE_SETS[feature_set_name]
    for params in LOGREG_PARAM_GRID:
        summary_row, fold_rows = evaluate_logreg_config_walk_forward(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            modeled_input_df=modeled_df,
            fold_specs=walk_forward_fold_specs,
        )
        walk_forward_grid_rows.append(summary_row)
        walk_forward_fold_rows.extend(fold_rows)

walk_forward_grid_results_df = pd.DataFrame(walk_forward_grid_rows)
walk_forward_fold_results_df = pd.DataFrame(walk_forward_fold_rows)
if selection_metric not in walk_forward_grid_results_df.columns:
    raise KeyError(f"Selection metric is not available: {selection_metric}")

validation_grid_results_df = walk_forward_grid_results_df.sort_values(
    [selection_metric, "balanced_accuracy", "f1_score", "accuracy", "feature_set", "param_set"],
    ascending=[False, False, False, False, True, True],
).reset_index(drop=True)


C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\user\anaconda3\envs\equity-price-direction-predictor\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The ma

In [10]:
validation_best_by_feature_set_df = ModelReportBuilder.select_best_validation_by_feature_set(
    validation_grid_results_df,
    selection_metric=CONFIG["selection_metric"],
)

validation_best_by_feature_set_report_df = ModelReportBuilder.build_validation_best_by_feature_set_report(
    validation_best_by_feature_set_df,
    param_columns=LOGREG_PARAM_COLUMNS_FOR_DISPLAY,
)

validation_best_by_feature_set_report_df


,feature_family,feature_set,n_features,param_set,penalty,C,solver,class_weight,l1_ratio,max_iter,validation_accuracy,validation_balanced_accuracy,validation_f1_score,validation_f1_weighted
0,price + volume + GDELT,Model C - price + volume + GDELT | GDELT zscor...,11,logreg_l1_C0p1_balanced_saga,l1,0.1,saga,balanced,NaN,5000,0.348047,0.373193,0.328229,0.331180
1,price + volume + GDELT + Reddit,Model D - price + volume + GDELT + Reddit | cl...,13,logreg_l2_C0p1_balanced_lbfgs,l2,0.1,lbfgs,balanced,NaN,3000,0.348437,0.372615,0.328848,0.331477
2,price + volume + GDELT,Model C - price + volume + GDELT | GDELT perce...,11,logreg_l2_C1_balanced_lbfgs,l2,1.0,lbfgs,balanced,NaN,3000,0.347266,0.371984,0.327891,0.330505
3,price + volume + GDELT,Model C - price + volume + GDELT | GDELT senti...,11,logreg_elasticnet_C1_l1ratio0p25_balanced_saga,elasticnet,1.0,saga,balanced,0.25,5000,0.348047,0.371678,0.328781,0.332152
4,price + volume + GDELT + Reddit,Model D - price + volume + GDELT + Reddit | zs...,13,logreg_elasticnet_C1_l1ratio0p25_balanced_saga,elasticnet,1.0,saga,balanced,0.25,5000,0.348047,0.371654,0.327854,0.331262
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,price + volume + Reddit sentiment lags + GDELT...,Model W - price + volume + Reddit sentiment la...,14,logreg_l1_C0p1_balanced_saga,l1,0.1,saga,balanced,NaN,5000,0.330078,0.351710,0.310633,0.313890
76,price + volume + GDELT + Google,Model H - price + volume + GDELT + Google | la...,12,logreg_l1_C0p1_balanced_saga,l1,0.1,saga,balanced,NaN,5000,0.325000,0.351325,0.303003,0.303163
77,price + volume + Google score attention,Model L - price + volume + Google score attent...,11,logreg_l2_C1_balanced_lbfgs,l2,1.0,lbfgs,balanced,NaN,3000,0.326562,0.351254,0.303575,0.305266
78,price + volume + GDELT sentiment lags + Google...,Model X - price + volume + GDELT sentiment lag...,14,logreg_l1_C0p1_balanced_saga,l1,0.1,saga,balanced,NaN,5000,0.326563,0.350684,0.305098,0.307635


In [11]:
best_validation_params_df = validation_best_by_feature_set_df.copy()


In [12]:
validation_refit_rows = []
test_rows = []

for row in best_validation_params_df.to_dict(orient="records"):
    params = params_from_result_row(row)
    feature_set_name = row["feature_set"]
    validation_refit_row = evaluate_logreg_params(
        feature_set_name=feature_set_name,
        features=FEATURE_SETS[feature_set_name],
        params=params,
        train_input_df=train_df,
        eval_input_df=validation_df,
        split_name="validation_refit_train",
    )
    validation_refit_rows.append(validation_refit_row)
    test_rows.append(
        evaluate_logreg_params(
            feature_set_name=feature_set_name,
            features=FEATURE_SETS[feature_set_name],
            params=params,
            train_input_df=train_validation_df,
            eval_input_df=test_df,
            split_name="test_refit_train_validation",
        )
    )

validation_refit_results_df = pd.DataFrame(validation_refit_rows)
test_best_validation_params_df = pd.DataFrame(test_rows).sort_values(
    ["balanced_accuracy", "f1_score", "accuracy", "feature_set"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

(
    simple_hyperparameter_summary_df,
    baseline_walk_forward_row,
    baseline_validation_refit_row,
    baseline_test_row,
) = ModelReportBuilder.build_simple_hyperparameter_summary(
    best_validation_params_df=best_validation_params_df,
    validation_refit_results_df=validation_refit_results_df,
    test_best_validation_params_df=test_best_validation_params_df,
    baseline_feature_set=BASELINE_FEATURE_SET,
)

In [13]:
validation_selected_family_test_report_df = ModelReportBuilder.build_validation_selected_family_test_report(
    simple_hyperparameter_summary_df,
    param_columns=LOGREG_PARAM_COLUMNS_FOR_DISPLAY,
)

final_test_verification_path = ModelReportBuilder.save_final_test_verification(
    validation_selected_family_test_report_df,
    model_name="logistic_regression",
    output_dir=PROJECT_ROOT / "notebooks" / "outputs-three-classes-rich-price",
)
print(f"Saved final test verification to: {final_test_verification_path}")

validation_selected_family_test_report_df


Saved final test verification to: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks\outputs-three-classes-rich-price\logistic_regression_final_test_verification.csv


,feature_family,feature_set,n_features,param_set,penalty,C,solver,class_weight,l1_ratio,max_iter,validation_accuracy,validation_balanced_accuracy,validation_f1_score,validation_f1_weighted,test_accuracy,test_balanced_accuracy,test_f1_score,test_f1_weighted,price_volume_baseline_feature_set,price_volume_baseline_test_balanced_accuracy,test_balanced_accuracy_change_vs_price_volume
0,price + volume + GDELT + Reddit,Model D - price + volume + GDELT + Reddit | cl...,13,logreg_l2_C0p1_balanced_lbfgs,l2,0.10,lbfgs,balanced,NaN,3000,0.348437,0.372615,0.328848,0.331477,0.353435,0.393491,0.345124,0.334817,Model B - price + volume | volume zscore 60d,0.384845,0.008646
1,price + volume + GDELT attention lags,Model S - price + volume + GDELT attention lag...,12,logreg_l1_C0p1_balanced_saga,l1,0.10,saga,balanced,NaN,5000,0.325781,0.352989,0.303912,0.304436,0.348642,0.392502,0.336744,0.323785,Model B - price + volume | volume zscore 60d,0.384845,0.007657
2,price + volume + GDELT sentiment + Reddit atte...,Model O - price + volume + GDELT sentiment + R...,11,logreg_l1_C0p1_balanced_saga,l1,0.10,saga,balanced,NaN,5000,0.332422,0.359479,0.313227,0.314328,0.348642,0.391597,0.337516,0.325034,Model B - price + volume | volume zscore 60d,0.384845,0.006751
3,price + volume + GDELT,Model C - price + volume + GDELT | GDELT zscor...,11,logreg_l1_C0p1_balanced_saga,l1,0.10,saga,balanced,NaN,5000,0.348047,0.373193,0.328229,0.331180,0.349840,0.390667,0.340801,0.329755,Model B - price + volume | volume zscore 60d,0.384845,0.005822
4,price + volume + Reddit sentiment lags + GDELT...,Model W - price + volume + Reddit sentiment la...,14,logreg_l1_C0p1_balanced_saga,l1,0.10,saga,balanced,NaN,5000,0.330469,0.353697,0.310818,0.313731,0.345447,0.389795,0.333139,0.319271,Model B - price + volume | volume zscore 60d,0.384845,0.004950
5,price + volume + GDELT + Reddit attention,Model M - price + volume + GDELT + Reddit atte...,11,logreg_l2_C0p01_balanced_lbfgs,l2,0.01,lbfgs,balanced,NaN,3000,0.342969,0.368307,0.321567,0.323158,0.346645,0.387703,0.337234,0.326042,Model B - price + volume | volume zscore 60d,0.384845,0.002858
6,price + volume + Reddit attention,Model K - price + volume + Reddit attention | ...,10,logreg_l2_C0p01_balanced_lbfgs,l2,0.01,lbfgs,balanced,NaN,3000,0.336719,0.363490,0.316020,0.317383,0.344649,0.387344,0.333726,0.321120,Model B - price + volume | volume zscore 60d,0.384845,0.002498
7,price + volume + Reddit sentiment lags + Googl...,Model Y - price + volume + Reddit sentiment la...,14,logreg_l1_C0p1_balanced_saga,l1,0.10,saga,balanced,NaN,5000,0.334766,0.357616,0.317744,0.321481,0.343850,0.387283,0.332404,0.318980,Model B - price + volume | volume zscore 60d,0.384845,0.002438
8,price only,Model A - price only,8,logreg_l2_C10_balanced_lbfgs,l2,10.00,lbfgs,balanced,NaN,3000,0.330859,0.359709,0.308364,0.306910,0.344649,0.387221,0.333396,0.320839,Model B - price + volume | volume zscore 60d,0.384845,0.002376
9,price + volume + Google attention lags,Model U - price + volume + Google attention la...,12,logreg_l1_C0p1_balanced_saga,l1,0.10,saga,balanced,NaN,5000,0.328516,0.353002,0.307785,0.310205,0.343450,0.386578,0.332049,0.319188,Model B - price + volume | volume zscore 60d,0.384845,0.001733
